<a href="https://colab.research.google.com/github/saikirannetha05/ultima_zepto_ai/blob/datapipeline/ultimacapstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import sqlite3
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import pandas as pd
import requests

# Suppress downcasting warnings
pd.set_option("future.no_silent_downcasting", True)


def scrape_books():
  """Scrapes books across nearly 8 different categories from books.toscrape.com"""
  base_url = "http://books.toscrape.com/"
  response = requests.get(base_url)
  response.raise_for_status()
  soup = BeautifulSoup(response.text, "html.parser")

  # Extract category links from the sidebar
  category_tags = soup.select(".side_categories ul li ul li a")
  categories = []
  for tag in category_tags:
    cat_name = tag.text.strip()
    cat_url = urljoin(base_url, tag["href"])
    categories.append({"name": cat_name, "url": cat_url})

  scraped_data = []
  # Target at least 8 categories to ensure a large, robust dataset (>= 60-150+ books)
  target_categories = categories[:8]
  print(f"Targeting {len(target_categories)} categories for scraping...")

  for cat in target_categories:
    cat_response = requests.get(cat["url"])
    if cat_response.status_code != 200:
      continue
    cat_soup = BeautifulSoup(cat_response.text, "html.parser")
    books = cat_soup.select("article.product_pod")

    for book in books:
      title = book.h3.a["title"]
      price_raw = book.select_one("p.price_color").text.strip()
      rating_class = book.p["class"]
      rating_raw = rating_class[1] if len(rating_class) > 1 else "Zero"
      availability_raw = book.select_one(
          "p.instock.availability"
      ).text.strip()

      scraped_data.append({
          "title": title,
          "price_raw": price_raw,
          "rating_raw": rating_raw,
          "availability_raw": availability_raw,
          "category": cat["name"],
      })

  return pd.DataFrame(scraped_data)


def clean_and_transform(df):
  """Cleans fields, handles parsing anomalies, and computes INR pricing."""
  def parse_price(val):
    try:
      return float(str(val).replace("£", "").strip())
    except Exception:
      return None

  df["price_gbp"] = df["price_raw"].apply(parse_price)

  median_price = df["price_gbp"].median()
  df["price_gbp"] = df["price_gbp"].fillna(median_price)

  rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

  def parse_rating(val):
    return rating_map.get(str(val).strip(), None)

  df["rating"] = df["rating_raw"].apply(parse_rating)
  median_rating = int(df["rating"].median())
  df["rating"] = df["rating"].fillna(median_rating).astype(int)

  def parse_availability(val):
    text = str(val).lower()
    return "in stock" in text

  df["in_stock"] = df["availability_raw"].apply(parse_availability)

  # Fixed project baseline: 1 GBP = 105.50 INR
  FIXED_CONVERSION_RATE = 105.50
  df["price_inr"] = df["price_gbp"] * FIXED_CONVERSION_RATE

  df = df[["title", "price_gbp", "price_inr", "rating", "in_stock", "category"]]
  return df


def load_to_sqlite(df, db_path="zepto_catalog.db"):
  """Designs normalized schema and loads data into SQLite."""
  conn = sqlite3.connect(db_path)
  cursor = conn.cursor()

  cursor.execute("""
        CREATE TABLE IF NOT EXISTS categories (
            category_id INTEGER PRIMARY KEY AUTOINCREMENT,
            category_name TEXT UNIQUE
        )
    """)

  cursor.execute("""
        CREATE TABLE IF NOT EXISTS books (
            book_id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            price_gbp REAL,
            price_inr REAL,
            rating INTEGER,
            in_stock INTEGER,
            category_id INTEGER,
            FOREIGN KEY (category_id) REFERENCES categories(category_id)
        )
    """)
  conn.commit()

  unique_categories = df["category"].unique()
  for cat in unique_categories:
    cursor.execute(
        "INSERT OR IGNORE INTO categories (category_name) VALUES (?)", (cat,)
    )
  conn.commit()

  cursor.execute("SELECT category_id, category_name FROM categories")
  cat_mapping = {name: cid for cid, name in cursor.fetchall()}

  df = df.copy()
  df["category_id"] = df["category"].map(cat_mapping)

  books_to_insert = df[[
      "title",
      "price_gbp",
      "price_inr",
      "rating",
      "in_stock",
      "category_id",
  ]].values.tolist()

  cursor.executemany(
      """
        INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?)
      """,
      books_to_insert,
  )
  conn.commit()
  conn.close()
  print(
      f"Successfully loaded {len(books_to_insert)} records into SQLite database"
      f" ({db_path})."
  )


def run_sql_queries(db_path="zepto_catalog.db"):
  """Executes the 5 required SQL queries covering all specified clauses."""
  conn = sqlite3.connect(db_path)

  print("\n--- QUERY 1: DISTINCT Categories ---")
  q1 = "SELECT DISTINCT category_name FROM categories ORDER BY category_name ASC;"
  print(pd.read_sql(q1, conn))

  print("\n--- QUERY 2: WHERE & BETWEEN & LIMIT (Price Range) ---")
  q2 = (
      "SELECT title, price_gbp FROM books WHERE price_gbp BETWEEN 10.00 AND"
      " 20.00 ORDER BY price_gbp DESC LIMIT 5;"
  )
  print(pd.read_sql(q2, conn))

  print("\n--- QUERY 3: IN Clause (Ratings 4 or 5) ---")
  q3 = "SELECT title, rating FROM books WHERE rating IN (4, 5) LIMIT 5;"
  print(pd.read_sql(q3, conn))

  print("\n--- QUERY 4: ORDER BY DESC (Highest INR Price) ---")
  q4 = "SELECT title, price_inr FROM books ORDER BY price_inr DESC LIMIT 5;"
  print(pd.read_sql(q4, conn))

  print("\n--- QUERY 5: JOIN (Highest-Rated Books per Category) ---")
  q5 = """
        SELECT b.title, c.category_name, b.rating, b.price_inr
        FROM books b
        JOIN categories c ON b.category_id = c.category_id
        WHERE b.rating = 5
        LIMIT 5;
    """
  df_sql_join = pd.read_sql(q5, conn)
  print(df_sql_join)

  conn.close()
  return df_sql_join


def verify_pandas_merge(db_path="zepto_catalog.db"):
  """Verifies that pd.read_sql join and pd.merge produce equivalent outputs."""
  conn = sqlite3.connect(db_path)
  df_books = pd.read_sql("SELECT * FROM books", conn)
  df_cats = pd.read_sql("SELECT * FROM categories", conn)
  conn.close()

  merged_df = pd.merge(df_books, df_cats, on="category_id")
  filtered_merged = merged_df[merged_df["rating"] == 5][
      ["title", "category_name", "rating", "price_inr"]
  ].head(5)

  print("\n--- PANDAS MERGE VERIFICATION OUTPUT ---")
  print(filtered_merged)


if __name__ == "__main__":
  print("Step 1: Scraping live catalog data across 8 categories...")
  raw_df = scrape_books()
  print(f"Scraped {len(raw_df)} total records across multiple categories.")

  print("Step 2: Cleaning and converting data...")
  cleaned_df = clean_and_transform(raw_df)

  print("Step 3: Loading into normalized SQLite schema...")
  load_to_sqlite(cleaned_df)

  print("Step 4: Executing required SQL queries...")
  run_sql_queries()

  print("Step 5: Running pandas merge validation...")
  verify_pandas_merge()

Step 1: Scraping live catalog data across 8 categories...
Targeting 8 categories for scraping...
Scraped 138 total records across multiple categories.
Step 2: Cleaning and converting data...
Step 3: Loading into normalized SQLite schema...
Successfully loaded 138 records into SQLite database (zepto_catalog.db).
Step 4: Executing required SQL queries...

--- QUERY 1: DISTINCT Categories ---
        category_name
0            Classics
1  Historical Fiction
2             Mystery
3          Philosophy
4             Romance
5      Sequential Art
6              Travel
7      Womens Fiction

--- QUERY 2: WHERE & BETWEEN & LIMIT (Price Range) ---
Empty DataFrame
Columns: [title, price_gbp]
Index: []

--- QUERY 3: IN Clause (Ratings 4 or 5) ---
                                               title  rating
0  Full Moon over Noahâs Ark: An Odyssey to Mou...       4
1                   A Year in Provence (Provence #1)       4
2                 1,000 Places to See Before You Die       5
3         

/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
